### OCR Benchmark Evaluation — OmniDocBench EN

Đánh giá 3 model OCR (**`EasyOCR`**, **`PP-OCRv6_medium`**, **`Surya OCR`**) trên **OmniDocBench EN** (cố định Layout = `DocLayout-YOLO`, `OcrMode.FULL_PAGE`, Dynamic Image Scaling).

* **`NED` *(Normalized Edit Distance)*:** Khoảng cách chỉnh sửa chuẩn hóa (`Levenshtein(pred, gt) / max(|pred|, |gt|)`), càng thấp càng tốt.
* **`Similarity` *(`1 - NED`)*:** Độ tương đồng ký tự theo phần trăm (`(1 - NED) × 100%`), càng cao càng tốt.
* **`Micro CER` *(Character Error Rate)*:** Tỷ lệ lỗi ký tự tổng trên toàn tập (`∑ Levenshtein / ∑ |gt| × 100%`), càng thấp càng tốt.
* **`Len Ratio (Pred/GT)`:** Tỷ lệ tổng số ký tự dự đoán trên đáp án (`≈ 1.0` là cân bằng; `< 1` đọc sót, `> 1` đọc thừa).

In [1]:
import json
import re
import unicodedata
from collections import defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display

BASE_DIR = Path(".").resolve()
GT_PATH = BASE_DIR / "ground_truth.json"
MODELS = ["EasyOCR", "PP-OCRv6_medium", "Surya OCR"]
EXCLUDE_GT_CATS = {"header", "footer", "page_number"}
EXCLUDE_PRED_LABELS = {"page_header", "page_footer"}


def levenshtein_distance(a: str, b: str) -> int:
    """Exact Levenshtein distance via Myers' bit-parallel algorithm."""
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    if len(a) > len(b):
        a, b = b, a
    m = len(a)
    peq: dict[str, int] = {}
    for i, ch in enumerate(a):
        peq[ch] = peq.get(ch, 0) | (1 << i)
    pv = (1 << m) - 1
    mv = 0
    score = m
    hb = 1 << (m - 1)
    mask = (1 << m) - 1
    for ch in b:
        eq = peq.get(ch, 0)
        xv = eq | mv
        xh = (((eq & pv) + pv) ^ pv) | eq
        ph = mv | (~(xh | pv) & mask)
        mh = pv & xh
        if ph & hb:
            score += 1
        elif mh & hb:
            score -= 1
        ph = ((ph << 1) | 1) & mask
        mh = (mh << 1) & mask
        pv = mh | (~(xv | ph) & mask)
        mv = ph & xv
    return score


def normalize_en(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    return re.sub(r"\s+", " ", text).strip()


def extract_gt_text(page: dict) -> str:
    items = []
    for d in page.get("layout_dets", []):
        if d.get("ignore"):
            continue
        if d.get("category_type") in EXCLUDE_GT_CATS:
            continue
        txt = (d.get("text") or "").strip()
        if not txt:
            continue
        order = d.get("order")
        poly = d.get("poly") or [0] * 8
        key = (0, order) if order is not None else (1, min(poly[1::2]), min(poly[0::2]))
        items.append((key, txt))
    items.sort(key=lambda x: x[0])
    return normalize_en("\n".join(t for _, t in items))


def extract_pred_text(doc: dict) -> str:
    parts = []
    for t in doc.get("texts", []):
        if t.get("label") in EXCLUDE_PRED_LABELS:
            continue
        txt = (t.get("text") or "").strip()
        if txt:
            parts.append(txt)
    return normalize_en("\n".join(parts))


gt_list = json.loads(GT_PATH.read_text(encoding="utf-8"))
gt_map = {Path(p["page_info"]["image_path"]).stem: p for p in gt_list}

records = []
for model in MODELS:
    m_dir = BASE_DIR / model
    for f in sorted(m_dir.glob("*.json")):
        if f.name in {"errors.json", "current_files.json"}:
            continue
        page = gt_map.get(f.stem)
        if page is None:
            continue
        gt_str = extract_gt_text(page)
        if not gt_str:
            continue
        pred_doc = json.loads(f.read_text(encoding="utf-8"))
        pred_str = extract_pred_text(pred_doc)
        ed = levenshtein_distance(pred_str, gt_str)
        ned = ed / max(len(pred_str), len(gt_str))
        attrs = page.get("page_info", {}).get("page_attribute", {})
        records.append({
            "model": model,
            "file": f.stem,
            "data_source": attrs.get("data_source", "unknown"),
            "layout": attrs.get("layout", "unknown"),
            "gt_len": len(gt_str),
            "pred_len": len(pred_str),
            "ed": ed,
            "ned": ned,
            "sim": (1.0 - ned) * 100.0,
        })

df = pd.DataFrame(records)
print(f"Evaluated {df['file'].nunique()} pages x {df['model'].nunique()} models = {len(df)} rows")

Evaluated 747 pages x 3 models = 2241 rows


In [2]:
# Table 1: Overall OCR Performance
overall_rows = []
for model in MODELS:
    sub = df[df["model"] == model]
    overall_rows.append({
        "Model": model,
        "Pages": len(sub),
        "NED (↓)": round(sub["ned"].mean(), 4),
        "Similarity 1-NED % (↑)": round(sub["sim"].mean(), 2),
        "Micro CER % (↓)": round(sub["ed"].sum() / sub["gt_len"].sum() * 100.0, 2),
        "Len Ratio (Pred/GT)": round(sub["pred_len"].sum() / sub["gt_len"].sum(), 3),
    })
overall_df = pd.DataFrame(overall_rows).sort_values("NED (↓)").reset_index(drop=True)
display(overall_df)

,Model,Pages,NED (↓),Similarity 1-NED % (↑),Micro CER % (↓),Len Ratio (Pred/GT)
0,Surya OCR,747,0.2793,72.07,31.86,1.156
1,PP-OCRv6_medium,747,0.2884,71.16,34.11,1.035
2,EasyOCR,747,0.4590,54.10,48.36,0.800


In [3]:
# Table 2: Per PDF Type (data_source) — Similarity (1-NED %) and NED
counts = df[df["model"] == MODELS[0]].groupby("data_source")["file"].count()
piv_sim = df.pivot_table(index="data_source", columns="model", values="sim", aggfunc="mean")[MODELS].round(2)
piv_ned = df.pivot_table(index="data_source", columns="model", values="ned", aggfunc="mean")[MODELS].round(4)

per_type = pd.DataFrame({"Pages": counts})
for m in MODELS:
    per_type[f"{m} Sim% (↑)"] = piv_sim[m]
for m in MODELS:
    per_type[f"{m} NED (↓)"] = piv_ned[m]
per_type = per_type.sort_values("Pages", ascending=False)
display(per_type)

,Pages,EasyOCR Sim% (↑),PP-OCRv6_medium Sim% (↑),Surya OCR Sim% (↑),EasyOCR NED (↓),PP-OCRv6_medium NED (↓),Surya OCR NED (↓)
data_source,,,,,,,
academic_literature,205,47.93,63.40,64.19,0.5207,0.3660,0.3581
book,130,51.26,71.23,71.44,0.4874,0.2877,0.2856
PPT2PDF,118,63.19,77.16,75.11,0.3681,0.2284,0.2489
magazine,84,61.05,81.04,83.64,0.3895,0.1896,0.1636
exam_paper,81,52.84,77.41,73.09,0.4716,0.2259,0.2691
newspaper,77,57.18,65.25,79.15,0.4282,0.3475,0.2085
colorful_textbook,51,52.11,72.35,68.38,0.4789,0.2765,0.3162
research_report,1,2.08,2.13,2.01,0.9792,0.9787,0.9799


In [4]:
# Table 3: Per Column Layout Type
l_counts = df[df["model"] == MODELS[0]].groupby("layout")["file"].count()
piv_l_sim = df.pivot_table(index="layout", columns="model", values="sim", aggfunc="mean")[MODELS].round(2)
piv_l_ned = df.pivot_table(index="layout", columns="model", values="ned", aggfunc="mean")[MODELS].round(4)

per_layout = pd.DataFrame({"Pages": l_counts})
for m in MODELS:
    per_layout[f"{m} Sim% (↑)"] = piv_l_sim[m]
for m in MODELS:
    per_layout[f"{m} NED (↓)"] = piv_l_ned[m]
per_layout = per_layout.sort_values("Pages", ascending=False)
display(per_layout)

,Pages,EasyOCR Sim% (↑),PP-OCRv6_medium Sim% (↑),Surya OCR Sim% (↑),EasyOCR NED (↓),PP-OCRv6_medium NED (↓),Surya OCR NED (↓)
layout,,,,,,,
single_column,336,50.37,69.43,69.64,0.4963,0.3057,0.3036
other_layout,157,52.11,66.23,69.50,0.4789,0.3377,0.3050
double_column,105,63.44,82.40,80.89,0.3656,0.1760,0.1911
1andmore_column,101,55.60,70.68,68.76,0.4440,0.2932,0.3124
three_column,48,63.20,75.84,85.23,0.3680,0.2416,0.1477
